# 01 — Data Acquisition & Preparation

**STAT 5243 Project 4 — Team 6 (MLB Hall of Fame Prediction).**

This notebook is the auditable run-through of Part 1. It demonstrates four data sources, eight relational table merges, and the construction of the binary target. Outputs feed into Part 2 (EDA and clustering) and Part 4 (modeling).

**Inputs:**
- SABR / Lahman 1871–2025 — `data/raw/sabr_lahman/` (27 tables — primary source)
- Kaggle Baseball Databank — `data/raw/kaggle/` (backup audit copy)
- pybaseball samples — `data/raw/pybaseball/`
- BeautifulSoup scrape — `data/raw/scraped/`

**Outputs:**
- `data/processed/player_features_base.csv`
- Source manifests in `reports/tables/`


## 1. Setup


In [ ]:
import sys
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import build_player_features
from src.config import DATA_RAW_KAGGLE, DATA_PROCESSED, TABLES

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
print("Project root:", PROJECT_ROOT)


## 2. Verify all four data sources are present

We expect the SABR/Lahman 2025 CSVs to be the primary source, with Kaggle, pybaseball, and BeautifulSoup data also on disk.


In [ ]:
sources = {
    "SABR/Lahman 2025": PROJECT_ROOT / "data/raw/sabr_lahman",
    "Kaggle":            PROJECT_ROOT / "data/raw/kaggle",
    "pybaseball":        PROJECT_ROOT / "data/raw/pybaseball",
    "BeautifulSoup":     PROJECT_ROOT / "data/raw/scraped",
    "Analysis input":    PROJECT_ROOT / "data/raw/analysis_input",
}
manifest = []
for name, path in sources.items():
    if path.exists():
        files = sorted([f.name for f in path.glob("*") if f.is_file()])
        manifest.append({"source": name, "path": str(path.relative_to(PROJECT_ROOT)),
                         "n_files": len(files), "files": files[:3]})
    else:
        manifest.append({"source": name, "path": str(path.relative_to(PROJECT_ROOT)),
                         "n_files": 0, "files": []})
pd.DataFrame(manifest)


## 3. Load all 14+ tables from analysis_input/


In [ ]:
raw_dir = PROJECT_ROOT / "data/raw/analysis_input"
core_files = ["Master.csv", "Batting.csv", "Pitching.csv", "Fielding.csv",
              "HallOfFame.csv", "AwardsPlayers.csv", "AwardsSharePlayers.csv",
              "AllstarFull.csv", "Salaries.csv", "Teams.csv",
              "BattingPost.csv", "PitchingPost.csv", "FieldingOF.csv", "SeriesPost.csv"]
optional_files = ["Appearances.csv"]

rows = []
for fname in core_files + optional_files:
    path = raw_dir / fname
    if path.exists():
        df = pd.read_csv(path, low_memory=False)
        rows.append({"table": fname, "rows": len(df), "columns": df.shape[1],
                     "missing_rate": round(df.isna().mean().mean(), 3),
                     "year_min": int(df["yearID"].min()) if "yearID" in df.columns else None,
                     "year_max": int(df["yearID"].max()) if "yearID" in df.columns else None})
    else:
        rows.append({"table": fname, "rows": "MISSING", "columns": None, "missing_rate": None,
                     "year_min": None, "year_max": None})

audit = pd.DataFrame(rows)
audit


## 4. Run the feature-building pipeline

This calls `build_player_features()` from `src/features.py`, which:
1. Aggregates batting and pitching across team stints
2. Merges all 14 tables on `playerID` / `(playerID, yearID)`
3. Computes era-adjusted metrics (OPS+, ERA+ proxies)
4. Adds 3-year and 5-year rolling peak features
5. Builds the `is_eligible` flag (BBWAA: ≥ 10 seasons + retired ≥ 5 years)
6. Constructs the `inducted` target


In [ ]:
df = build_player_features(raw_dir, DATA_PROCESSED)
print(f"Output: {df.shape[0]:,} player rows × {df.shape[1]} columns")
print(f"Eligible modeling pool: {int(df['model_eligible_pool'].sum()):,}")
print(f"Inducted: {int(df['inducted'].sum()):,}")
print(f"Year coverage: {int(df['debut_year'].min())}–{int(df['final_year'].max())}")


## 5. Class imbalance audit


In [ ]:
pool = df[df["model_eligible_pool"] == 1]
print(f"Eligible players       : {len(pool):>6,}")
print(f"  Inducted (positive)  : {int(pool['inducted'].sum()):>6,} ({pool['inducted'].mean()*100:.1f}%)")
print(f"  Not inducted         : {int((1-pool['inducted']).sum()):>6,}")

fig, ax = plt.subplots(figsize=(6, 4))
counts = pool["inducted"].value_counts().sort_index()
ax.bar(["Not inducted", "Inducted"], counts.values, color=["#888", "#C62828"])
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("Players"); ax.set_title("Hall of Fame target — class imbalance")
plt.tight_layout(); plt.show()


## 6. Spot-check legendary players (sanity check on joins)


In [ ]:
legends = ["ruthba01", "mayswi01", "aaronha01", "koufasa01", "jeterde01",
           "pujolal01", "griffke02", "ryanno01", "trouutmi01"]
cols = [c for c in ["playerID", "nameFirst", "nameLast", "primary_role", "primary_position",
                    "n_mlb_seasons", "is_eligible", "inducted",
                    "bat_HR", "bat_OPS", "pit_W", "pit_SO",
                    "allstar_games", "award_total"] if c in df.columns]
df[df["playerID"].isin(legends)][cols].sort_values("playerID")


## 7. Top-25 missingness audit


In [ ]:
miss = df.isna().mean().sort_values(ascending=False).head(25)
print("Top 25 columns by missingness:")
print(miss.to_string())

fig, ax = plt.subplots(figsize=(8, 6))
miss.plot(kind="barh", ax=ax, color="#C62828")
ax.invert_yaxis()
ax.set_xlabel("Missing rate")
ax.set_title("Top 25 columns by missingness")
plt.tight_layout(); plt.show()


## 8. Eligibility breakdown by era

Era distribution helps us understand whether the eligible-but-not-inducted population is balanced across history.


In [ ]:
if "debut_era" in df.columns:
    era_table = (df[df["model_eligible_pool"] == 1]
                 .groupby("debut_era")
                 .agg(eligible=("playerID", "count"),
                      inducted=("inducted", "sum"))
                 .assign(induction_rate=lambda x: (x["inducted"] / x["eligible"]).round(3))
                 .reset_index())
    print(era_table.to_string(index=False))


## 9. Summary KPI for the report


In [ ]:
kpis = {
    "n_players_total":          int(len(df)),
    "n_eligible_players":       int(pool.shape[0]),
    "n_inducted":               int(pool["inducted"].sum()),
    "induction_rate":           round(float(pool["inducted"].mean()), 4),
    "median_career_seasons":    float(pool["n_mlb_seasons"].median()),
    "year_range":               f"{int(df['debut_year'].min())}-{int(df['final_year'].max())}",
    "n_features":               int(df.shape[1]),
    "n_pitchers_eligible":      int((pool["primary_role"] == "Pitcher").sum()),
    "n_hitters_eligible":       int((pool["primary_role"] == "Hitter").sum()),
}
print(json.dumps(kpis, indent=2))


## 10. Hand-off

The processed file `data/processed/player_features_base.csv` is the input that Person B (EDA & clustering, Part 2) and Person C (feature engineering & modeling, Parts 3-4) consume. They do not need to re-touch raw CSVs.

```python
import pandas as pd
df = pd.read_parquet("data/processed/player_features_base.csv")  # or read_csv
```
